# Gold Layer — Aggregations
## NYC Yellow Taxi — January 2024

Reads from the Silver clean table and produces business-level aggregations.

**Source:** `nyc_taxi.silver.yellow_trips_clean`  
**Targets:**
- `nyc_taxi.gold.hourly_trip_volume`
- `nyc_taxi.gold.location_fare_stats`
- `nyc_taxi.gold.payment_type_distribution`

In [0]:
from pyspark.sql import functions as F

df = spark.table("nyc_taxi.silver.yellow_trips_clean")
df.count()

2752609

## Aggregation 1 — Hourly Trip Volume
When during the day are taxis most in demand?

In [0]:
hourly_volume = (df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour").agg(F.count("*").alias("trip_count")).orderBy("pickup_hour"))

hourly_volume.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.hourly_trip_volume")
display(hourly_volume.sort("trip_count", ascending=False))

pickup_hour,trip_count
18,197893
17,193226
16,180228
15,178916
14,173287
19,172307
13,161000
12,155353
20,150602
21,149690


Databricks visualization. Run in Databricks to view.

## Aggregation 2 — Average Fare & Tip by Pickup Location
Which pickup zones generate the highest fares and tips?

In [0]:
location_fare_stats = (df.groupBy("PULocationID").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance")
    ).orderBy("trip_count", ascending=False))

location_fare_stats.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.location_fare_stats")

In [0]:
display(location_fare_stats.sort("avg_fare", ascending=False))

PULocationID,trip_count,avg_fare,avg_tip,avg_distance
44,1,264.1,58.75,57.2
118,3,125.9,1.67,18.74
206,1,100.0,0.0,0.4
111,1,89.0,0.0,5.3
1,49,84.37,10.57,5.77
265,461,82.76,9.02,7.95
251,1,70.2,24.6,17.6
172,2,70.0,21.81,18.06
8,11,67.67,11.45,9.93
132,137937,62.86,9.19,15.94


Databricks visualization. Run in Databricks to view.

In [0]:
display(location_fare_stats.sort("avg_tip", ascending=False))


PULocationID,trip_count,avg_fare,avg_tip,avg_distance
44,1,264.1,58.75,57.2
251,1,70.2,24.6,17.6
172,2,70.0,21.81,18.06
8,11,67.67,11.45,9.93
1,49,84.37,10.57,5.77
176,1,26.8,10.0,6.22
132,137937,62.86,9.19,15.94
265,461,82.76,9.02,7.95
2,2,53.3,8.9,14.15
93,446,60.91,8.87,9.08


Databricks visualization. Run in Databricks to view.

## Aggregation 3 — Payment Type Distribution
How do passengers prefer to pay?

In [0]:
# Payment type codes per TLC data dictionary:
# 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided trip
payment_distribution = (df.groupBy("payment_type").agg(F.count("*").alias("trip_count"),F.round(F.avg("tip_amount"), 2).alias("avg_tip")
    ).orderBy("trip_count", ascending=False))

payment_distribution.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.payment_type_distribution")
display(payment_distribution)

payment_type,trip_count,avg_tip
1,2297063,4.16
2,422235,0.0
4,22755,0.0
3,10556,0.01


Databricks visualization. Run in Databricks to view.

## Key Findings

- **Credit card** is the dominant payment method at 83.45%, with ~17% of riders still paying cash
- **Zone 43 (JFK Airport)** is a clear outlier — highest avg fare ($264.10) and avg tip ($58.75), consistent with long-distance airport transfers
- **Peak demand** occurs at 18:00 with ~198K trips, reflecting NYC evening rush hour